In [2]:
# === 0) Paths y setup ===
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("synthetic_data")
DATA_PATH = ROOT / "synthetic_plant.csv"             # series originales (date_time + var_1..var_N)
MANUAL_PATH = ROOT / "etiquetado_manual.csv"         # tus etiquetas manuales (long o wide)
AUTO_MAIN   = ROOT / "synthetic_plant_labels.csv"    # etiquetas automáticas (si existe)
AUTO_FLAGS  = ROOT / "synthetic_plant_flags.csv"     # alternativa (si existe)

OUT_DIR = ROOT / "labels_manual" / "compare"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR ->", OUT_DIR)

OUT_DIR -> synthetic_data\labels_manual\compare


In [3]:
# === 1) Series ===
df = pd.read_csv(DATA_PATH, parse_dates=["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

var_cols = [c for c in df.columns if c != "date_time" and pd.api.types.is_numeric_dtype(df[c])]
assert len(var_cols) > 0, "No hay columnas numéricas en synthetic_plant.csv"
N_VARS = len(var_cols)
N_VARS, var_cols[:5]

(10, ['var_1', 'var_2', 'var_3', 'var_4', 'var_5'])

In [4]:
# === 2) Normalizadores (a wide con sufijo) ===
def to_wide_labels(df_lab: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """
    Convierte etiquetas a formato wide con columnas 'drift_var_i{suffix}'.
    Soporta:
      - long:  date_time, variable, label
      - wide:  date_time + {var_i}_label
      - wide:  date_time + drift_var_i
      - wide:  date_time + var_i (0/1)  -> lo tratamos como etiqueta
    """
    df_lab = df_lab.copy()
    df_lab["date_time"] = pd.to_datetime(df_lab["date_time"], errors="coerce")
    df_lab = df_lab.dropna(subset=["date_time"]).sort_values("date_time").reset_index(drop=True)

    cols = set(df_lab.columns)
    # Caso long
    if {"date_time", "variable", "label"}.issubset(cols):
        wide = (
            df_lab.pivot(index="date_time", columns="variable", values="label")
                  .reset_index()
                  .sort_values("date_time")
        )
    else:
        # Caso wide, renombrar columnas candidatas a etiquetas
        wide = df_lab.copy()
        # si ya existen *_label o drift_* las aceptamos
        # si existen {var} en 0/1, las tratamos como etiquetas
        # no tocamos 'date_time'
    # Estandarizar nombres -> drift_var_i{suffix}
    rename = {}
    for c in wide.columns:
        if c == "date_time": 
            continue
        base = c
        if base.endswith("_label"):
            base = base[:-6]
        base = base.replace("drift_", "")  # aceptar drift_var_i o var_i
        rename[c] = f"drift_{base}{suffix}"
    wide = wide.rename(columns=rename)

    # Forzar 0/1 enteros
    for c in wide.columns:
        if c == "date_time":
            continue
        wide[c] = wide[c].fillna(0)
        # si no es bool/int, intenta mapear {True,False} a 1/0 o castear
        try:
            wide[c] = (wide[c].astype(int) > 0).astype(int)
        except Exception:
            wide[c] = wide[c].astype(float).fillna(0.0)
            wide[c] = (wide[c] > 0.5).astype(int)
    return wide

def load_auto_labels():
    # intenta AUTO_MAIN, luego AUTO_FLAGS
    for p in [AUTO_MAIN, AUTO_FLAGS]:
        if p.exists():
            auto_raw = pd.read_csv(p, parse_dates=["date_time"])
            return to_wide_labels(auto_raw, suffix="_auto"), p.name
    raise FileNotFoundError("No encontré synthetic_plant_labels.csv ni synthetic_plant_flags.csv")

In [5]:
# === 3) Cargar manual y auto; merge con series ===
manual_raw = pd.read_csv(MANUAL_PATH, parse_dates=["date_time"])
manual_w   = to_wide_labels(manual_raw, suffix="_manual")

auto_w, auto_used = load_auto_labels()

merged = (
    df.merge(manual_w, on="date_time", how="left")
      .merge(auto_w,   on="date_time", how="left")
      .sort_values("date_time")
      .reset_index(drop=True)
)

# Completar NaN de etiquetas con 0
for v in var_cols:
    mcol = f"drift_{v}_manual"
    acol = f"drift_{v}_auto"
    if mcol in merged.columns: merged[mcol] = merged[mcol].fillna(0).astype(int)
    if acol in merged.columns: merged[acol] = merged[acol].fillna(0).astype(int)

# Variables comparables (manual y auto presentes)
vars_common = [v for v in var_cols if f"drift_{v}_manual" in merged.columns and f"drift_{v}_auto" in merged.columns]
assert vars_common, "No hay variables en común para comparar (manual vs auto)."
f"Auto usado: {auto_used} | Variables comparables: {len(vars_common)}"

ValueError: Missing column provided to 'parse_dates': 'date_time'

In [ ]:
# === 4) Métricas ===
def metrics_one(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    tp = int(((y_true==1) & (y_pred==1)).sum())
    tn = int(((y_true==0) & (y_pred==0)).sum())
    fp = int(((y_true==0) & (y_pred==1)).sum())
    fn = int(((y_true==1) & (y_pred==0)).sum())
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    acc  = (tp+tn)/max(1,(tp+tn+fp+fn))
    return tp, fp, tn, fn, prec, rec, f1, acc

rows = []
all_tp=all_fp=all_tn=all_fn=0
for v in vars_common:
    y_true = merged[f"drift_{v}_manual"]
    y_pred = merged[f"drift_{v}_auto"]
    tp, fp, tn, fn, prec, rec, f1, acc = metrics_one(y_true, y_pred)
    rows.append(dict(variable=v, TP=tp, FP=fp, TN=tn, FN=fn,
                     Precision=prec, Recall=rec, F1=f1, Accuracy=acc))
    all_tp+=tp; all_fp+=fp; all_tn+=tn; all_fn+=fn

metrics_df = pd.DataFrame(rows).sort_values("F1", ascending=False)

# Macro-avg (promedio entre variables)
macro = dict(
    Precision=metrics_df["Precision"].mean(),
    Recall=metrics_df["Recall"].mean(),
    F1=metrics_df["F1"].mean(),
    Accuracy=metrics_df["Accuracy"].mean(),
)

# Micro-avg (agregado sobre todas las muestras)
micro_prec = all_tp/(all_tp+all_fp) if (all_tp+all_fp)>0 else 0.0
micro_rec  = all_tp/(all_tp+all_fn) if (all_tp+all_fn)>0 else 0.0
micro_f1   = 2*micro_prec*micro_rec/(micro_prec+micro_rec) if (micro_prec+micro_rec)>0 else 0.0
micro_acc  = (all_tp+all_tn)/max(1,(all_tp+all_tn+all_fp+all_fn))

summary_df = pd.DataFrame([
    {"Averaging":"Macro", **{k: round(v,4) for k,v in macro.items()}},
    {"Averaging":"Micro", "Precision":round(micro_prec,4), "Recall":round(micro_rec,4),
     "F1":round(micro_f1,4), "Accuracy":round(micro_acc,4)}
])

metrics_df.to_csv(OUT_DIR/"metrics_per_variable.csv", index=False)
summary_df.to_csv(OUT_DIR/"metrics_summary.csv", index=False)

metrics_df.head(10), summary_df

In [ ]:
# === 5) Delay de detección (manual GT) ===
# Para cada episodio manual (bandas contiguas de 1), calcula el primer timestamp detectado por AUTO
# y reporta (t_pred - t_true_start) en minutos (positivos = auto detecta después).
delays = []

for v in vars_common:
    m = merged[["date_time", f"drift_{v}_manual", f"drift_{v}_auto"]].copy()
    m = m.sort_values("date_time")
    # detectar episodios manuales (runs de 1)
    inside = False
    start_idx = None
    for i, row in m.iterrows():
        cur = row[f"drift_{v}_manual"]
        if cur == 1 and not inside:
            inside = True
            start_idx = i
        if (cur == 0 and inside) or (inside and i == m.index[-1]):
            # episodio cerrado
            end_i = i if cur==0 else i  # cierre en el último índice 1
            episode = m.loc[start_idx:end_i]
            t0 = episode["date_time"].iloc[0]
            # primer detectado por auto dentro del episodio o después de t0 (con un margen)
            after = m[m["date_time"] >= t0]
            hit = after[after[f"drift_{v}_auto"] == 1]
            if len(hit):
                delay_min = (hit["date_time"].iloc[0] - t0).total_seconds()/60.0
            else:
                delay_min = np.nan  # nunca detectó
            delays.append(dict(variable=v, start=t0, detection_delay_min=delay_min))
            inside = False
            start_idx = None

delays_df = pd.DataFrame(delays).sort_values(["variable","start"])
delays_df.to_csv(OUT_DIR/"detection_delays.csv", index=False)
delays_df.head(10)

In [ ]:
# === 6) Plot comparativo (bandas: auto vs manual) ===
import matplotlib.pyplot as plt

def plot_compare(var, xmin=None, xmax=None, show_series=True):
    d = merged.copy()
    if xmin is not None:
        d = d[d["date_time"] >= pd.to_datetime(xmin)]
    if xmax is not None:
        d = d[d["date_time"] <= pd.to_datetime(xmax)]

    t = d["date_time"]
    y = d[var]
    a = d[f"drift_{var}_auto"].astype(bool).values
    m = d[f"drift_{var}_manual"].astype(bool).values

    plt.figure(figsize=(11,3.6))
    if show_series:
        plt.plot(t, y, lw=0.9, alpha=0.8, label=var)

    ax = plt.gca()
    ax.fill_between(t, 0, 1, where=a, alpha=0.22, transform=ax.get_xaxis_transform(), label="Auto")
    ax.fill_between(t, 0, 1, where=m, alpha=0.22, transform=ax.get_xaxis_transform(), label="Manual")

    # mini-métricas en título
    y_true = d[f"drift_{var}_manual"]; y_pred = d[f"drift_{var}_auto"]
    tp, fp, tn, fn, prec, rec, f1, acc = metrics_one(y_true, y_pred)
    plt.title(f"{var} | P={prec:.2f} R={rec:.2f} F1={f1:.2f} Acc={acc:.2f}")
    plt.xlabel("Tiempo"); plt.ylabel(var if show_series else "")
    plt.grid(True, alpha=0.3); plt.legend(loc="upper right")
    plt.tight_layout(); plt.show()

# ejemplo: grafica las 3 con peor F1 (para mirar rápido)
for v in metrics_df.sort_values("F1").head(3)["variable"]:
    plot_compare(v)

In [ ]:
# === 7) Export consolidados para reporte ===
# - merged reducido a etiquetas y tiempo
lab_cols = []
for v in vars_common:
    lab_cols += [f"drift_{v}_manual", f"drift_{v}_auto"]

labels_time = merged[["date_time"] + lab_cols].copy()
labels_time.to_csv(OUT_DIR/"labels_time_aligned.csv", index=False)

print("Guardado:")
print(" - metrics_per_variable.csv")
print(" - metrics_summary.csv")
print(" - detection_delays.csv")
print(" - labels_time_aligned.csv")